# Pydantic basics (Part 1)

Pydantic is the validation layer used across this whole training. Three ideas to take away:

1. **A model is a typed contract.** You declare fields with types; Pydantic builds the validator.
2. **Validation happens at construction.** Bad data raises immediately, near the source.
3. **Models serialize cleanly.** `model_dump()` / `model_validate_json()` move between Python and JSON.

In Part 1 we already use Pydantic in two places:
- `Config` / `TasksConfig` in `config/settings.py` (typed configuration).
- `ServerSummary` in `main.py` (structured LLM output).

Run the cells top to bottom.

In [ ]:
from pydantic import BaseModel


class Product(BaseModel):
    name: str
    price: float
    in_stock: bool


# Construction validates and coerces types where it is safe to do so.
p = Product(name="Chai", price="18.0", in_stock="yes")
print(p)
print(type(p.price), type(p.in_stock))

name='Chai' price=18.0 in_stock=True
<class 'float'> <class 'bool'>


## Validation errors are explicit

When data does not fit the contract, Pydantic raises a `ValidationError` that lists every problem at once.

In [2]:
from pydantic import ValidationError

try:
    Product(name="Tofu", price="cheap", in_stock=True)
except ValidationError as exc:
    print(exc)

1 validation error for Product
price
  Input should be a valid number, unable to parse string as a number [type=float_parsing, input_value='cheap', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/float_parsing


## Constraints with `Field`, secrets with `SecretStr`

`Field` adds rules (min length, ranges) to a type. `SecretStr` hides a value in logs and `repr`, exactly how the API key is handled in `config/settings.py`.

In [3]:
from pydantic import Field, SecretStr


class Credentials(BaseModel):
    models: list[str] = Field(min_length=1)
    api_key: SecretStr


creds = Credentials(models=["model-a"], api_key="sk-or-v1-secret")
print(creds)                          # api_key shows as **********
print(creds.api_key.get_secret_value())  # explicit read of the real value

models=['model-a'] api_key=SecretStr('**********')
sk-or-v1-secret


## Serialization: Python <-> JSON

`model_dump()` gives a dict, `model_dump_json()` a JSON string, and `model_validate_json()` parses JSON back into a validated model. This is exactly what the LLM client relies on when it repairs and re-validates structured output.

In [4]:
p = Product(name="Chai", price=18.0, in_stock=True)

print(p.model_dump())
print(p.model_dump_json())

# Parse JSON text straight into a validated object.
restored = Product.model_validate_json('{"name": "Chai", "price": 18.0, "in_stock": true}')
print(restored == p)

{'name': 'Chai', 'price': 18.0, 'in_stock': True}
{"name":"Chai","price":18.0,"in_stock":true}
True


## How Part 1 uses this: nested models for config

`tasks.yaml` is loaded into nested Pydantic models. A YAML dict becomes `TasksConfig` containing a dict of `TaskConfig`. If the file is malformed (e.g. a task with no models), validation fails at load time instead of at the first LLM call.

In [5]:
class TaskConfig(BaseModel):
    temperature: float
    models: list[str] = Field(min_length=1)


class TasksConfig(BaseModel):
    max_output_tokens: int
    tasks: dict[str, TaskConfig]


raw = {
    "max_output_tokens": 512,
    "tasks": {
        "precise": {"temperature": 0.0, "models": ["google/gemma-3-12b-it"]},
        "creative": {"temperature": 1.1, "models": ["google/gemma-3-12b-it"]},
    },
}

cfg = TasksConfig.model_validate(raw)
print(cfg.tasks["precise"].temperature, cfg.tasks["precise"].models)

0.0 ['google/gemma-3-12b-it']
